In [ ]:
!pip install autogen_agentchat autogen_ext --quiet

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

In [ ]:
import utils
import os

In [ ]:
model_client = OpenAIChatCompletionClient(model="gpt-4o-mini", api_key=os.environ['OPENAI_API_KEY'], base_url=os.environ['OPENAI_API_BASE'])

In [ ]:
PLANNER_SYS_PROMPT = "You are a helpful assistant that can suggest a travel plan for a user based on their request."

planner_agent = AssistantAgent(
    "planner_agent",
    model_client=model_client,
    description="A helpful assistant that can plan trips.",
    system_message= PLANNER_SYS_PROMPT
)

In [ ]:
LOCAL_SYS_PROMPT = """
You are a helpful assistant that can suggest authentic and interesting local activities or places
to visit for a user and can utilize any context information provided.
"""
local_agent = AssistantAgent(
    "local_agent",
    model_client=model_client,
    description="A local assistant that can suggest local activities or places to visit.",
    system_message= LOCAL_SYS_PROMPT
)

In [ ]:
LANG_SYS_PROMPT = """
"You are a helpful assistant that can review travel plans, providing feedback on important/critical tips about
how best to address language or communication challenges for the given destination.

If the plan already includes language tips, you can mention that the plan is satisfactory, with rationale.
"""
language_agent = AssistantAgent(
    "language_agent",
    model_client=model_client,
    description="A helpful assistant that can provide language tips for a given destination.",
    system_message=LANG_SYS_PROMPT
)

In [ ]:
TRAVEL_SYS_PROMPT = """
You are a helpful assistant that can take in all of the suggestions and advice from the other agents
and provide a detailed final travel plan.
You must ensure that the final plan is integrated and complete.

YOUR FINAL RESPONSE MUST BE THE COMPLETE PLAN.
When the plan is complete and all perspectives are integrated, you can respond with TERMINATE.
"""
travel_summary_agent = AssistantAgent(
    "travel_summary_agent",
    model_client=model_client,
    description="A helpful assistant that can summarize the travel plan.",
    system_message=TRAVEL_SYS_PROMPT
)

In [ ]:
termination = TextMentionTermination("TERMINATE")
group_chat = RoundRobinGroupChat(
    [planner_agent, local_agent, language_agent, travel_summary_agent], termination_condition=termination
)
await Console(group_chat.run_stream(task="Plan a 3 day trip to Goa."))

await model_client.close()